# 03 - YOLO11s training on PPE-5 (Kaggle)

Dataset: `/kaggle/input/datasets/duydq206/ppe5-data/ppe-5/data`.
Class IDs and names are read from this dataset's `data.yaml`; no class remapping is applied.
Run All on Kaggle with a GPU to train a new model. Outputs have been cleared.
Existing reports and `models/yolo11s/best.pt` belong to the previous training run.


## 1. Cài đặt Ultralytics

In [ ]:
import subprocess
import sys
# Giữ PyTorch/CUDA của Kaggle; ghi lại version thực tế trong báo cáo cuối.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3.0,<9'])


## 2. Import thư viện và kiểm tra GPU

In [ ]:
import os
import json
import shutil
from pathlib import Path
from datetime import datetime, timezone
import yaml
import torch
import ultralytics
import pandas as pd
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops, get_num_params
from IPython.display import display, FileLink, Image

WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
ALLOW_CPU = False  # Chỉ bật khi chủ động chấp nhận train chậm trên CPU.
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError('Bật Accelerator GPU trong Kaggle Settings, restart session rồi Run All.')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Dataset path

In [ ]:
# Use exactly the dataset requested for this run.
data_path = '/kaggle/input/datasets/duydq206/ppe5-data/ppe-5/data'
data_root = Path(data_path).resolve()
yaml_path = str(data_root / 'data.yaml')
if not Path(yaml_path).is_file():
    raise FileNotFoundError(
        f'Missing {yaml_path}. Attach ppe5-data in Kaggle Input and check the ppe-5/data folder.'
    )
print('Dataset:', data_root)


## 4. Kiểm tra cấu trúc dataset

In [ ]:
# Check the selected dataset before reading its split configuration.
assert data_root.is_dir(), f'Dataset directory does not exist: {data_root}'
print('Available split directories:', sorted(p.name for p in data_root.iterdir() if p.is_dir()))


## 5. Đọc data.yaml và sửa path cho phiên Kaggle hiện tại

Cell này giữ nguyên class mapping, chỉ tạo file YAML tạm trỏ tới đúng dataset. `train` và `val` có thể là đường dẫn tương đối hoặc tuyệt đối.

In [ ]:
with open(yaml_path, encoding='utf-8') as f:
    data_config = yaml.safe_load(f)
assert isinstance(data_config, dict) and all(k in data_config for k in ('train', 'val', 'names'))
print(yaml.safe_dump(data_config, allow_unicode=True, sort_keys=False))
raw_names = data_config['names']
class_names = ({int(k): str(v) for k, v in raw_names.items()}
               if isinstance(raw_names, dict) else dict(enumerate(raw_names)))
assert sorted(class_names) == list(range(len(class_names))) and class_names, 'Class IDs phải liên tục từ 0.'
if 'nc' in data_config:
    assert int(data_config['nc']) == len(class_names), 'nc không khớp names.'

def resolve_split(value, split):
    if not isinstance(value, str):
        raise ValueError(f'{split}: expected an image directory path.')
    root = Path(data_path).resolve()
    normalized = value.replace('\\', '/')
    declared = Path(normalized)
    candidates = []
    if not declared.is_absolute():
        candidates.append(root / declared)
    # Relocate stale Kaggle/Roboflow paths into the NEW selected dataset first.
    parts = normalized.split('/')
    for marker in ('train', 'valid', 'val', 'test'):
        if marker in parts:
            candidates.append(root.joinpath(*parts[parts.index(marker):]))
    candidates.append(declared if declared.is_absolute() else root / declared)
    for candidate in candidates:
        resolved = candidate.resolve()
        # Never accept a split from another mounted dataset, even if it exists.
        if resolved.is_relative_to(root) and resolved.is_dir():
            return str(resolved)
    raise FileNotFoundError(f'{split}: cannot resolve {value!r} inside {root}. Tried: {candidates}')

train_images_path = resolve_split(data_config['train'], 'train')
val_images_path = resolve_split(data_config['val'], 'val')
train_config = dict(data_config)
train_config.update(path=data_path, train=train_images_path, val=val_images_path)
# Không cho YAML tự tải/chạy bước download ngoài workflow dataset đã attach.
train_config.pop('download', None)
if train_config.get('test'):
    train_config['test'] = resolve_split(train_config['test'], 'test')
else:
    train_config.pop('test', None)
# Validate annotation directories for the splits declared by this dataset.
for split in ('train', 'val', 'test'):
    if split in train_config:
        labels_dir = Path(train_config[split]).parent / 'labels'
        if not labels_dir.is_dir():
            raise FileNotFoundError(f'{split}: missing label directory {labels_dir}')
if Path(train_images_path) == Path(val_images_path):
    raise ValueError('Train and validation must use different directories.')

train_config_path = str(WORK_DIR / 'ppe_train.yaml')
with open(train_config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(train_config, f, allow_unicode=True, sort_keys=False)
print('Training YAML:', train_config_path)
print('Classes:', class_names)


## 6. Đếm ảnh train và validation

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
def image_files(folder):
    return sorted(p for p in Path(folder).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
train_files = image_files(train_images_path)
val_files = image_files(val_images_path)
train_count, val_count = len(train_files), len(val_files)
print('Train images:', train_count)
print('Validation images:', val_count)
assert train_count > 0 and val_count > 0


## 7. Load pretrained YOLO11s

In [ ]:
model = YOLO('yolo11s.pt')
print('YOLO11s loaded successfully.')

## 8. Train YOLO11

Baseline: 100 epochs, image size 640, batch 16. Nếu GPU hết bộ nhớ, giảm batch xuống 8 hoặc 4.

In [ ]:
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

results = model.train(
    data=train_config_path,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    project='/kaggle/working/runs',
    name='ppe_yolo11s',
    pretrained=True,
    patience=20,
    save=True,
    plots=True,
    seed=42
)

## 9. Kiểm tra model đã lưu

In [ ]:
# Lấy thư mục thực tế: Ultralytics tự tăng hậu tố khi chạy lại.
run_dir = Path(model.trainer.save_dir)
best_model_path = run_dir / 'weights' / 'best.pt'
last_model_path = run_dir / 'weights' / 'last.pt'
print('Best model:', best_model_path, best_model_path.is_file())
print('Last model:', last_model_path, last_model_path.is_file())
assert best_model_path.is_file(), 'Không tìm thấy best.pt. Kiểm tra log train.'


## 10. Đánh giá best model trên validation

In [ ]:
def collect_detection_metrics(metrics, network, image_size):
    # get_flops returns GFLOPs directly; zero signals missing THOP or a profiling failure.
    params = int(get_num_params(network))
    gflops = float(get_flops(network, imgsz=image_size))
    if not gflops > 0:
        raise RuntimeError('Cannot profile GFLOPs. Check ultralytics-thop and rerun this cell.')
    f1_values = metrics.box.f1
    speed = {stage: float(metrics.speed[stage])
             for stage in ('preprocess', 'inference', 'postprocess')}
    return {
        'params': params,
        'params_millions': params / 1e6,
        'gflops': gflops,
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'f1': float(sum(f1_values) / len(f1_values)) if len(f1_values) else 0.0,
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        'latency_inference_ms': speed['inference'],
        'latency_preprocess_ms': speed['preprocess'],
        'latency_postprocess_ms': speed['postprocess'],
        'latency_pipeline_ms': sum(speed.values()),
    }


best_model = YOLO(best_model_path)
best_model.fuse()  # Profile the same fused model that validation uses.
metrics = best_model.val(
    data=train_config_path,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    half=False,
    plots=True,
    project='/kaggle/working/runs',
    name='ppe_yolo11s_validation'
)
evaluation = collect_detection_metrics(metrics, best_model.model, IMAGE_SIZE)
print(f"Params (fused): {evaluation['params']:,} ({evaluation['params_millions']:.3f} M)")
print(f"GFLOPs at imgsz={IMAGE_SIZE}: {evaluation['gflops']:.3f}")
print(f"Precision: {evaluation['precision']:.4f}")
print(f"Recall: {evaluation['recall']:.4f}")
print(f"F1 (macro): {evaluation['f1']:.4f}")
print(f"mAP50: {evaluation['mAP50']:.4f}")
print(f"mAP50-95: {evaluation['mAP50_95']:.4f}")
print(f"Latency inference (validation, batch={BATCH_SIZE}): {evaluation['latency_inference_ms']:.3f} ms/image")
print(f"Preprocess + inference + postprocess: {evaluation['latency_pipeline_ms']:.3f} ms/image")


## 11. mAP50 theo từng class

In [ ]:
# ap50 chỉ chứa class được đánh giá; ap_class_index ánh xạ về class ID thật.
ap_by_class = {int(cid): float(ap) for cid, ap in zip(metrics.box.ap_class_index, metrics.box.ap50)}
per_class = pd.DataFrame([
    {'class_id': cid, 'class_name': name, 'mAP50': ap_by_class.get(cid, float('nan'))}
    for cid, name in sorted(class_names.items())
])
display(per_class)
print('NaN: class không có AP được đánh giá; không tự gán 0.')
validation_dir = Path(metrics.save_dir)
per_class.to_csv(validation_dir / 'per_class_map50.csv', index=False)
metrics_table = pd.DataFrame([{
    'Model': 'YOLO11s',
    'Params (M)': evaluation['params_millions'],
    'GFLOPs': evaluation['gflops'],
    'P': evaluation['precision'],
    'R': evaluation['recall'],
    'F1': evaluation['f1'],
    'mAP50': evaluation['mAP50'],
    'mAP50-95': evaluation['mAP50_95'],
    'Latency (ms/img)': evaluation['latency_inference_ms'],
}])
display(metrics_table.round(4))
metrics_table.to_csv(validation_dir / 'metrics_summary.csv', index=False)
summary = {
    **evaluation,
    'params_model_state': 'fused',
    'flops_input_shape': [1, 3, IMAGE_SIZE, IMAGE_SIZE],
    'f1_definition': 'Mean per-class F1 at the Ultralytics max-mean-F1 operating point (IoU 0.5)',
    'latency_method': 'Validation average per image; pipeline = preprocess + inference + postprocess; excludes file I/O',
    'latency_batch_size': BATCH_SIZE,
    'latency_precision': 'FP32',
    'device': str(DEVICE),
    'device_name': torch.cuda.get_device_name(DEVICE) if DEVICE != 'cpu' else 'CPU',
    'train_images': train_count, 'val_images': val_count,
    'dataset_path': data_path, 'class_names': class_names, 'split': 'val',
    'model': 'yolo11s.pt', 'ultralytics': ultralytics.__version__,
    'torch': torch.__version__, 'epochs_requested': EPOCHS,
    'imgsz': IMAGE_SIZE, 'batch': BATCH_SIZE, 'seed': 42,
    'train_dir': str(run_dir), 'validation_dir': str(validation_dir),
    'created_utc': datetime.now(timezone.utc).isoformat()
}
with open(validation_dir / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
for plot in (run_dir / 'results.png', validation_dir / 'confusion_matrix.png'):
    if plot.is_file():
        display(Image(filename=str(plot), width=900))


## 12. Predict ảnh validation

In [ ]:
# stream=True: không giữ tất cả Results trong RAM; phải duyệt generator để chạy predict.
prediction_dir = None
prediction_count = 0
for prediction in best_model.predict(
    source=[str(p) for p in val_files],
    imgsz=IMAGE_SIZE, conf=0.5, device=DEVICE,
    save=True, stream=True,
    project='/kaggle/working/runs', name='ppe_yolo11s_predictions'
):
    prediction_dir = Path(prediction.save_dir)
    prediction_count += 1
assert prediction_count == val_count and prediction_dir is not None
print('Predicted images:', prediction_count)
print('Prediction outputs:', prediction_dir)
for preview in image_files(prediction_dir)[:4]:
    display(Image(filename=str(preview), width=700))


## 13. Output cần lưu

ZIP gồm thư mục train thực tế (best.pt, last.pt, results.csv, plots), validation (confusion matrix, summary.json, per_class_map50.csv), prediction và YAML đã chuẩn hóa.
Tên thư mục có thể được Ultralytics thêm hậu tố khi chạy lại; notebook dùng đường dẫn thực tế, không lấy nhầm checkpoint cũ.
Save Version trên Kaggle và giữ output để tải ZIP trong tab Output. FileLink cũng được cung cấp bên dưới.


In [ ]:
import zipfile
zip_path = WORK_DIR / f'{run_dir.name}_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in (run_dir, validation_dir, prediction_dir):
        for file in sorted(folder.rglob('*')):
            if file.is_file():
                archive.write(file, arcname=str(Path(folder.name) / file.relative_to(folder)))
    archive.write(train_config_path, arcname='ppe_train.yaml')
    archive.writestr('environment.txt', subprocess.check_output(
        [sys.executable, '-m', 'pip', 'freeze'], text=True))
print(f'Đã nén: {zip_path} ({zip_path.stat().st_size / 1024**2:.2f} MB)')
display(FileLink(str(zip_path.relative_to(WORK_DIR))))
